# S06 · Speculative decoding: exactness, payoff and confidence-scheduled verification

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 12. Sources: Leviathan et al. (arXiv:2211.17192); github.com/deepseek-ai/DeepSpec (7.2k stars, MIT: EAGLE-3, DFlash, DSpark); DSpark (arXiv:2607.05147).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Check that speculative sampling preserves the target distribution, map the speedup surface, and
model the newest idea: when drafters propose whole blocks, verify only the confident part.

## Theory and equations
Accept $x\sim q$ with $\min(1, p/q)$; else sample $\operatorname{norm}(\max(0,p-q))$.
$\mathbb{E}[\text{tokens}] = (1-\alpha^{\gamma+1})/(1-\alpha)$; speedup $= \mathbb{E}/(\gamma c + 1)$.
With position-dependent acceptance, verifying draft $j$ is worth $\prod_{i\le j}\alpha_i$ tokens; under a shared
slot budget, verify the drafts with the largest cumulative confidence.

## Diagram
```
block drafter (DFlash / DSpark):  t1 t2 t3 t4 t5 t6 t7  in one draft pass; confidence decays ->
verify:  request A (confident): t1..t6      request B (unsure): t1..t2      same total slots
```

## Implementation

In [ ]:
from inference_lab import specdec as SD
gen = torch.Generator().manual_seed(0)
p = torch.tensor([0.45, 0.25, 0.15, 0.10, 0.05]); q = torch.full((5,), 0.2)
counts = torch.zeros(5); acc = 0; n = 50_000
for _ in range(n):
    x, a = SD.speculative_step(p, q, gen); counts[x] += 1; acc += a
print("target p    :", p.tolist()); print("emitted (MC):", (counts / n).round(decimals=4).tolist())
print("acceptance MC", acc / n, " theory 1-TV", SD.acceptance_rate(p, q))

## Experiment 1: expected tokens, formula vs Monte Carlo

In [ ]:
for a in (0.5, 0.7, 0.9):
    print(a, [(g, round(SD.expected_tokens(a, g), 3), round(SD.simulate_chain(a, g, 100_000), 3)) for g in (2, 4, 8)])

## Experiment 2: confidence-scheduled vs fixed-length verification (*theoretical*)

In [ ]:
gen = torch.Generator().manual_seed(3)
N, gamma = 64, 8
base = torch.rand(N, 1, generator=gen) * 0.45 + 0.5                # per-request draft quality
alphas = (base * torch.linspace(1.0, 0.8, gamma)).clamp(max=0.99)    # acceptance decays along the block
for slots_per_req in (2, 3, 4, 6, 9):
    budget = N * slots_per_req
    lengths, tot = SD.confidence_schedule(alphas, budget)
    k, tot_fixed = SD.fixed_schedule(alphas, budget)
    print(f"{slots_per_req} slots/request: fixed k={k} -> {tot_fixed:6.1f} tokens; "
          f"confidence-scheduled -> {tot:6.1f} tokens (lengths {int(lengths.min())}..{int(lengths.max())})")

## Interpretation
The emitted distribution matches $p$. Under a tight slot budget, giving long verifications to
confident requests and short ones to unsure requests yields more tokens per batch than a
uniform draft length. That is the scheduling problem DSpark addresses for high-concurrency serving.

## Limitations
I.i.d. or fixed per-position acceptance; real acceptance depends on content.

## Conclusion
Speculation never changes outputs. At high concurrency, what you verify matters as much as what you draft.